# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and analyze the FAIR² dataset ([Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)) using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset is described via a Croissant schema available at:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant --quiet

## 1. Data Loading

We load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}\n")
print(f"Version: {metadata.version}\nLicense: {metadata.license}\n")
print(f"Identifier: {metadata.identifier}\nPublished: {metadata.datePublished}\n")

## 2. Data Overview

Let's review available record sets and fields, identified by their `@id` fields.

In [ ]:
# List the record sets and fields in the metadata by `@id`.

record_set_ids = []
print("Available Record Sets (@id and name):")
for record_set in metadata.record_sets:
    record_set_ids.append(record_set.id)
    print(f" - @id: {record_set.id}    name: {record_set.name}")

    # Print fields/columns in this record set
    print("   Fields/Columns in this Record Set:")
    for field in record_set.fields:
        print(f"     - @id: {field.id}   name: {field.name}   dataType: {field.data_type if hasattr(field,'data_type') else 'N/A'}")
    print()

if not record_set_ids:
    print("No record sets were found in the metadata. Please check the dataset schema.")

## 3. Data Extraction

We extract data from each record set into pandas DataFrames. You can select the relevant record set(s) using their `@id` as found above.

In [ ]:
# Extract data from each record set identified above
# We'll load all available record sets

dataframes = {}
for record_set_id in record_set_ids:
    print(f"\nLoading records for record set @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"  -> Loaded {len(df)} records.\n  Columns: {list(df.columns)}")

if record_set_ids:
    # Pick the first record set for demonstration
    primary_record_set_id = record_set_ids[0]
    print("\nPreview of records from record set '", primary_record_set_id, "':")
    display(dataframes[primary_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)

Next, we'll perform basic EDA: filtering, normalization, and grouping. Adjust the `numeric_field_id` and `group_field_id` below using valid `@id` values (see the columns above).

In [ ]:
# EDA Example: Filter, Normalize, Group
import numpy as np

# If there is at least one record set loaded, proceed
if record_set_ids:
    df = dataframes[primary_record_set_id].copy()
    print(f"Working with record set: {primary_record_set_id}. Available columns:")
    print(list(df.columns))

    # Attempt to automatically pick a numeric field (`@id`) for demonstration
    # If none available, skip numeric analysis
    numeric_field_id = None
    candidate_types = ['Float', 'Number', 'Integer']
    for record_set in metadata.record_sets:
        if record_set.id == primary_record_set_id:
            for field in record_set.fields:
                if hasattr(field, 'data_type') and field.data_type in candidate_types:
                    if field.id in df.columns:
                        numeric_field_id = field.id
                        break
            break
    if numeric_field_id is not None and numeric_field_id in df.columns:
        print(f"Using numeric field (@id): {numeric_field_id}")
        # Remove NA for demonstration
        df_numeric = pd.to_numeric(df[numeric_field_id], errors='coerce')
        threshold = np.nanmean(df_numeric)
        filtered_df = df[df_numeric > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id].astype(float) - df_numeric.mean()) / df_numeric.std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try to group by a categorical field present in DataFrame
        # Pick a different field than the numeric one
        group_field_id = None
        for record_set in metadata.record_sets:
            if record_set.id == primary_record_set_id:
                for field in record_set.fields:
                    if field.id != numeric_field_id and field.id in df.columns:
                        # Select first non-numeric field
                        if getattr(field, 'data_type', None) not in candidate_types:
                            group_field_id = field.id
                            break
                break
        if group_field_id:
            print(f"Grouping by field (@id): {group_field_id}")
            # Only group if grouping field is categorical, skip NA
            grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print(f"Mean of {numeric_field_id} grouped by {group_field_id}:")
            display(grouped)
    else:
        print("No numeric field (Float/Number/Integer) detected for EDA in this record set.")
else:
    print("No record sets available for analysis.")

## 5. Visualization

Visualize the distribution of a numeric field using a histogram, and the relationship between the numeric field and a group (if available) using a boxplot.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

if record_set_ids and numeric_field_id and numeric_field_id in df:
    fig, axs = plt.subplots(1,2, figsize=(14,5))
    # Histogram
    axs[0].hist(pd.to_numeric(df[numeric_field_id], errors='coerce').dropna(), bins=12, color='skyblue', edgecolor='k')
    axs[0].set_title(f'Distribution of {numeric_field_id}')
    axs[0].set_xlabel(numeric_field_id)
    axs[0].set_ylabel('Count')
    # Boxplot by group (if present)
    if group_field_id and group_field_id in df:
        sns.boxplot(data=df, x=group_field_id, y=pd.to_numeric(df[numeric_field_id], errors='coerce'), ax=axs[1])
        axs[1].set_title(f'{numeric_field_id} by {group_field_id}')
    plt.tight_layout()
    plt.show()
else:
    print("No numeric field for visualization.")

## 6. Conclusion

In this notebook, we explored the FAIR² dataset using `mlcroissant`, loaded records, identified fields and record sets by their `@id`, and performed basic analysis and visualizations. You can extend this notebook by applying domain-specific analysis relevant to clinicopathological studies in oncology or by exploring other record sets and fields.

**Note:** All dataset entities are referenced by their Croissant `@id` for clarity and reproducibility.